# Module 32 — Instruction Fine-Tuning

Everything through Module 31 is **pretraining**: predict the next token,
everywhere, on raw text. That produces a model that can continue text
plausibly, but has no notion of "answer this question" as a distinct
behavior — it just keeps generating whatever seems statistically likely to
follow. **Instruction fine-tuning (SFT)** teaches a specific behavior on
top: given a (prompt, response) pair, only compute the loss on the
**response** tokens — the model isn't penalized for how well it
"predicts" the prompt (which it doesn't need to generate, since the prompt
is given), only for producing the right response after it.

This module pretrains a small model on Module 18's toy corpus (same setup,
not re-explained), then instruction-fine-tunes it on a handful of
Question/Answer pairs, and proves the loss-masking mechanism is correct
before checking what actually changed.

## 1. Pretraining phase (same setup as Module 18)

In [ ]:
import copy
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

CORPUS = """Aether and Lumine are twins known as the Traveler. They came from another world and lost each other upon arrival in Teyvat. Paimon found Aether floating near Mondstadt and decided to travel together. Mondstadt is called the City of Freedom, and the wind blows gently across its hills. Klee loves to explore the city and often causes small explosions with her bombs. Diluc runs the Dawn Winery outside the city walls. Kaeya works at the Knights of Favonius and enjoys teasing Diluc. Jean leads the Knights of Favonius with great responsibility. Barbara sings songs at the church and heals the sick. Venti wanders the city playing his lyre and humming old songs. Amber flies her glider over the fields, scouting for trouble.

In Liyue, the harbor bustles with merchants and travelers from every nation. Zhongli walks slowly through the streets, remembering old stories. Xiao guards Liyue from atop Mount Aocang, watching the clouds. Ningguang watches the city from her floating Jade Chamber. Xiangling cooks delicious food and feeds her friend Guoba. Hu Tao manages the Wangsheng Funeral Parlor with a playful smile. Beidou sails her ship across the sea, laughing at the storm. Keqing trains alone at night, sharp and determined. Ganyu works quietly at the Yuehai Pavilion, tired but dutiful.

In Inazuma, thunder rumbles over the scattered islands. The Raiden Shogun seeks eternity within her quiet domain. Kazuha travels with the wind, calm and free, never staying long. Yae Miko runs a shop and enjoys teasing visitors who wander in. Ayaka trains in the snow near Inazuma City, graceful and composed. Yoimiya sets off fireworks that light up the night sky. Itto challenges anyone brave enough to an arm-wrestling match.

In Sumeru, the forest hums with ancient knowledge and old machines. Tighnari studies the creatures of the rainforest with careful eyes. Nahida watches over the dreams of Sumeru from the Akademiya. Cyno enforces the law with a solemn face, rarely smiling. Dehya guards the desert routes, strong and steady under the sun.

In Fontaine, the water sparkles under the courtroom lights. Furina performs on stage before the Opera Epiclese, dramatic and bright. Neuvillette presides over the Palais Mermonia with quiet authority. Wriothesley guards the Fortress of Meropide far below the sea. Lyney performs magic tricks that delight the crowd every evening.

Travelers explore each nation, meeting new friends and solving old mysteries. Every journey begins with a single step through the city gate. Paimon is always hungry and asks about food more than anything else. The wind carries stories from one nation to the next, never resting."""

PROMPT_MARKER = "\n### Answer:"
qa_examples = [
    ("### Question: Who is Klee?", " Klee loves to explore the city and often causes small explosions with her bombs."),
    ("### Question: Who is Zhongli?", " Zhongli walks slowly through the streets, remembering old stories."),
    ("### Question: What does Paimon do?", " Paimon is always hungry and asks about food more than anything else."),
    ("### Question: Who is Venti?", " Venti wanders the city playing his lyre and humming old songs."),
]
test_prompt = "### Question: Who is Diluc?" + PROMPT_MARKER  # held out - not one of the 4 SFT examples above
extra_chars = "".join(q + PROMPT_MARKER + a for q, a in qa_examples) + test_prompt

torch.manual_seed(42)
chars = sorted(set(CORPUS) | set(extra_chars))  # the base vocab plus whatever the SFT template/examples need
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)
data = torch.tensor([stoi[c] for c in CORPUS], dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

BLOCK_SIZE = 32          # pretraining context window (Module 18)
MAX_SEQ_LEN = 150        # the model\'s position-embedding capacity needs headroom for the longer SFT examples below
batch_size = 32
device = "cuda" if torch.cuda.is_available() else "cpu"

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(0, len(d) - BLOCK_SIZE - 1, (batch_size,))
    x = torch.stack([d[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([d[i + 1:i + 1 + BLOCK_SIZE] for i in ix])
    return x.to(device), y.to(device)

def scaled_dot_product_attention(Q, K, V, causal=True):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        seq_len_q, seq_len_k = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len_q, seq_len_k, device=scores.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

def split_heads(t, num_heads):
    *batch_dims, seq_len, d_model = t.shape
    d_k = d_model // num_heads
    return t.view(*batch_dims, seq_len, num_heads, d_k).transpose(-3, -2)

def merge_heads(t):
    *batch_dims, num_heads, seq_len, d_k = t.shape
    return t.transpose(-3, -2).contiguous().view(*batch_dims, seq_len, num_heads * d_k)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x, causal=True):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        Qh, Kh, Vh = split_heads(Q, self.num_heads), split_heads(K, self.num_heads), split_heads(V, self.num_heads)
        out, _ = scaled_dot_product_attention(Qh, Kh, Vh, causal=causal)
        return self.Wo(merge_heads(out))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.ln1(x), causal=True)
        x = x + self.ffn(self.ln2(x))
        return x

class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len, d_ff=None):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_embed.weight
    def forward(self, idx):
        seq_len = idx.shape[-1]
        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)


torch.manual_seed(42)
model = NanoGPT(vocab_size, d_model=64, num_heads=4, num_layers=4, max_seq_len=MAX_SEQ_LEN).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

@torch.no_grad()
def estimate_val_loss(iters=10):
    model.eval()
    losses = torch.zeros(iters)
    for i in range(iters):
        x, y = get_batch("val")
        losses[i] = F.cross_entropy(model(x).view(-1, vocab_size), y.view(-1)).item()
    model.train()
    return losses.mean().item()

best_val, best_state = float("inf"), None
for step in range(400):
    x, y = get_batch("train")
    loss = F.cross_entropy(model(x).view(-1, vocab_size), y.view(-1))
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    if step % 50 == 0:
        vl = estimate_val_loss()
        if vl < best_val:
            best_val, best_state = vl, copy.deepcopy(model.state_dict())
        print(f"step {step:3d}   train {loss.item():.3f}   val {vl:.3f}   best {best_val:.3f}")

model.load_state_dict(best_state)  # the best (least-overfit) checkpoint, per Module 18\'s lesson
print(f"\nLoaded best pretrained checkpoint (val loss {best_val:.3f}).")

## 2. Building instruction/response pairs with a loss mask

Wrap each (question, answer) as `"### Question: ...\n### Answer: ..."` and
mark, per token position, whether it's part of the **response** (loss
counts) or the **prompt** (loss ignored) — the actual mechanism instruction
fine-tuning depends on.

In [ ]:
def encode(s):
    return [stoi[c] for c in s]

sft_examples = []
for question, answer in qa_examples:
    prompt_text = question + PROMPT_MARKER
    full_ids = encode(prompt_text + answer)
    prompt_len = len(encode(prompt_text))
    sft_examples.append((full_ids, prompt_len))

max_len = max(len(ids) for ids, _ in sft_examples)
input_batch = torch.zeros((len(sft_examples), max_len - 1), dtype=torch.long)
target_batch = torch.zeros((len(sft_examples), max_len - 1), dtype=torch.long)
mask_batch = torch.zeros((len(sft_examples), max_len - 1))

for i, (ids, plen) in enumerate(sft_examples):
    L = len(ids) - 1
    input_batch[i, :L] = torch.tensor(ids[:-1])
    target_batch[i, :L] = torch.tensor(ids[1:])
    for t in range(L):
        if (t + 1) >= plen:  # target position t predicts ids[t+1]; count it only if that\'s in the response
            mask_batch[i, t] = 1.0

input_batch, target_batch, mask_batch = input_batch.to(device), target_batch.to(device), mask_batch.to(device)
print(f"{len(sft_examples)} SFT examples, response-token fraction: {mask_batch.mean().item():.1%}")

## 3. Verifying the masked loss against a manual per-example computation

The vectorized masked loss (used for actual training, since it processes
the whole batch at once) must equal manually slicing out just each
example's response tokens and computing ordinary cross-entropy on them.

In [ ]:
def masked_loss(logits, targets, mask):
    losses = F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1), reduction="none")
    losses = losses.view(targets.shape)
    return (losses * mask).sum() / mask.sum()

with torch.no_grad():
    logits = model(input_batch)

manual_losses = []
for i, (ids, plen) in enumerate(sft_examples):
    L = len(ids) - 1
    resp_logits = logits[i, plen - 1:L]
    resp_targets = target_batch[i, plen - 1:L]
    manual_losses.append(F.cross_entropy(resp_logits, resp_targets, reduction="sum").item())
manual_total = sum(manual_losses) / mask_batch.sum().item()
vectorized_total = masked_loss(logits, target_batch, mask_batch).item()

print(f"manual masked loss:     {manual_total:.6f}")
print(f"vectorized masked loss: {vectorized_total:.6f}")
assert abs(manual_total - vectorized_total) < 1e-4
print("Confirmed: the vectorized masked loss exactly matches computing loss only on each example\'s response tokens.")

## 4. Before vs. after: what actually changed

The held-out test prompt asks about **Diluc** — deliberately not one of
the 4 SFT examples, so the answer isn't something SFT directly taught.

In [ ]:
@torch.no_grad()
def generate(m, prompt, max_new_chars=70):
    idx = torch.tensor([encode(prompt)], device=device)
    for _ in range(max_new_chars):
        context = idx[:, -MAX_SEQ_LEN:]
        probs = F.softmax(m(context)[:, -1, :], dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_idx], dim=1)
    return "".join(itos[i] for i in idx[0].tolist())


print("BEFORE instruction fine-tuning:")
print(repr(generate(model, test_prompt)))

In [ ]:
sft_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
for step in range(100):
    logits = model(input_batch)
    loss = masked_loss(logits, target_batch, mask_batch)
    sft_optimizer.zero_grad()
    loss.backward()
    sft_optimizer.step()
    if step % 20 == 0:
        print(f"sft step {step:3d}   loss {loss.item():.4f}")

print("\nAFTER instruction fine-tuning:")
print(repr(generate(model, test_prompt)))

## Recap

- Instruction fine-tuning's core mechanism — computing loss only on
  response tokens, not the prompt — was verified exactly against a manual
  per-example computation.
- **Before** fine-tuning: given `"### Question: Who is Diluc?\n### Answer:"`,
  the base pretrained model (which never saw this template meaningfully,
  since those position indices went mostly untrained during pretraining)
  produced incoherent text after the marker.
- **After** fine-tuning on just 4 unrelated Q/A pairs: the same held-out
  prompt about Diluc produced a grammatically real, properly-formatted
  sentence immediately after `"### Answer:"` — borrowed from one of the
  memorized training answers, since Diluc specifically wasn't taught, but
  demonstrating exactly what SFT is supposed to teach: the *behavior* of
  answering in the expected format, which is a separate thing from having
  the specific facts. With only 4 tiny examples, heavy memorization
  (train loss to ~0.0005) is expected, same lesson as Modules 09 and 18 —
  real instruction tuning needs many more examples to generalize facts,
  not just format.

Module 33 (stretch) covers RLHF/DPO concepts — the further alignment step
beyond supervised fine-tuning that production conversational models use.